# Notebook 4 of 4: Churn model tracked with MLflow (Part 6)

The MLflow version that comes with Databricks serverless failed for me on `set_experiment`, with the error `Configuration spark.mlflow.modelRegistryUri is not available`. Upgrading MLflow fixed it. After a pip install Python has to restart, which is why the imports are in the next cell.

In [ ]:
%pip install -q --upgrade mlflow scikit-learn
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CATALOG = "workspace"
SCHEMA = "mustafa_shop"
DB = f"{CATALOG}.{SCHEMA}"

CUTOFF = "2026-08-01"
LABEL_END = "2026-10-24"

# Databricks can turn on autologging by itself. It would log every small fit inside
# cross-validation, so I switch it off and log only what I choose.
mlflow.autolog(disable=True)

user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT = f"/Users/{user}/online_shop_churn"
mlflow.set_experiment(EXPERIMENT)

2026/11/04 15:41:27 INFO mlflow.tracking.fluent: Experiment with name '/Users/<my_email>/online_shop_churn' does not exist. Creating a new experiment.

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1551877586757301', creation_time=1793806887481, effective_trace_archival_retention=None, experiment_id='1551877586757301', last_update_time=1793806887481, lifecycle_stage='active', name='/Users/<my_email>/online_shop_churn', tags={'mlflow.experiment.sourceName': '/Users/<my_email>/online_shop_churn',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': '<my_email>',
 'mlflow.ownerId': '71868155914640'}, trace_location=None, workspace='default'>


### 6.1 Features, only from orders before the cutoff
I do **not** reuse `gold_customer_features`, because it was built from all orders up to October and that would be leakage. Everything here is rebuilt from orders dated before 2026-08-01, and `days_since_last_order` is counted back from the cutoff.

I also do not use `status` as a feature. An order placed in July can become "returned" in August, so today's status can contain information from after the cutoff.

In [ ]:
orders = (spark.table(f"{DB}.fact_orders")
    .join(spark.table(f"{DB}.dim_customer").select("customer_sk", "customer_id"), "customer_sk")
    .join(spark.table(f"{DB}.dim_date").select("date_key", "date"), "date_key"))

before = orders.filter(F.col("date") < CUTOFF)
after = orders.filter((F.col("date") >= CUTOFF) & (F.col("date") <= LABEL_END))

cutoff_date = F.to_date(F.lit(CUTOFF))
features = (before
    .groupBy("customer_id")
    .agg(F.count("*").alias("num_orders"),
         F.sum("amount").alias("total_spend"),
         F.avg("amount").alias("avg_order_value"),
         F.countDistinct("category_key").alias("num_categories"),
         F.avg(F.when(F.col("source") == "app", 1).otherwise(0)).alias("app_order_share"),
         F.max("date").alias("last_order_date"),
         F.sum(F.when(F.col("date") >= F.date_sub(cutoff_date, 90), 1).otherwise(0)).alias("orders_last_90_days"))
    .withColumn("days_since_last_order", F.datediff(cutoff_date, F.col("last_order_date"))))

### 6.2 Labels
The brief's label: **churned = 1 if the customer placed no order between 2026-08-01 and 2026-10-24.** Any order counts, whatever its status, because the customer still came back to buy.

In [ ]:
orders_after = after.groupBy("customer_id").agg(F.count("*").alias("orders_in_window"))

training = (features
    .join(orders_after, "customer_id", "left")
    .fillna({"orders_in_window": 0})
    .withColumn("churned", (F.col("orders_in_window") == 0).cast("int"))
    .withColumn("went_quiet", (F.col("orders_in_window") <= 2).cast("int")))

# Leakage check: no feature can come from the cutoff date or later
latest = training.agg(F.max("last_order_date")).first()[0]
print("Latest order used in the features:", latest)
assert str(latest) < CUTOFF

training.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.gold_churn_training")

# The Delta version gets logged with every run, so I know which data each model saw
features_version = spark.sql(f"DESCRIBE HISTORY {DB}.gold_churn_training").agg(F.max("version")).first()[0]
print("Features table version:", features_version)

data = spark.table(f"{DB}.gold_churn_training").toPandas().sort_values("customer_id").reset_index(drop=True)
print(f"Customers: {len(data):,}")
print(data[["churned", "went_quiet"]].sum())
print(data["orders_in_window"].value_counts().sort_index().head(6))

Latest order used in the features: 2026-07-31
Features table version: 0
Customers: 2,000
churned        2
went_quiet    83
dtype: int64
0      2
1     28
2     53
3     96
4    162
5    165
Name: orders_in_window, dtype: int64


### A problem with the label
Only **2 of the 2,000 customers** have no order in the window. Customers in this data order often (the smallest count before the cutoff is 20 orders), so almost nobody goes 85 days without buying. This affects the whole of Part 6:

- A model that says "nobody churns" scores 99.9% accuracy and is useless, so accuracy tells us nothing here.
- A normal 75/25 split put both churned customers in the training data and left none to test on.
- Two examples are not enough to learn from. Below you can see both had ordered in the week before the cutoff, so they looked like any other active customer.

**What I did:**
1. I kept the brief's label and logged three runs on it, as asked.
2. I added a second, wider label so the marketing team can get something usable: `went_quiet` = at most 2 orders in the 85-day window. This is my own choice and not in the brief, and I kept the two clearly separate in MLflow.

In [ ]:
print(data.loc[data["churned"] == 1, ["customer_id", "num_orders", "days_since_last_order", "orders_last_90_days"]].to_string(index=False))
print("\nA typical customer, for comparison:")
print(data[["num_orders", "days_since_last_order", "orders_last_90_days"]].median().to_string())

 customer_id  num_orders  days_since_last_order  orders_last_90_days
          51          32                      7                    7
        1174          30                      6                    6

A typical customer, for comparison:
num_orders               55.0
days_since_last_order     7.0
orders_last_90_days       9.0


### 6.3 Train and log the models
I use cross-validation because the positive cases are so few. Every customer gets predicted once by a model that did not train on them, and the metrics come from those predictions. With 2 positives there can only be 2 folds, which is the only way to test on each one. After that, the logged model is fitted on all the data.

`class_weight="balanced"` makes a missed positive cost more. Without it the models simply learn to say 0 every time.

In [ ]:
FEATURES = ["num_orders", "total_spend", "avg_order_value", "num_categories", "app_order_share",
            "days_since_last_order", "orders_last_90_days"]


def make_models():
    return {
        "logistic_regression": (make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)), None),
        "random_forest_depth4": (RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42), 4),
        "random_forest_depth8": (RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42), 8),
    }


X = data[FEATURES]
results = []

for label in ["churned", "went_quiet"]:
    y = data[label]
    folds = StratifiedKFold(n_splits=min(5, y.sum()), shuffle=True, random_state=42)

    for model_name, (model, depth) in make_models().items():
        with mlflow.start_run(run_name=f"{label}__{model_name}") as run:
            scores = cross_val_predict(model, X, y, cv=folds, method="predict_proba")[:, 1]
            predicted = (scores >= 0.5).astype(int)

            metrics = {
                "accuracy": accuracy_score(y, predicted),
                "recall": recall_score(y, predicted, zero_division=0),
                "precision": precision_score(y, predicted, zero_division=0),
                "f1": f1_score(y, predicted, zero_division=0),
                "roc_auc": roc_auc_score(y, scores),
                "customers_flagged": int(predicted.sum()),
            }
            mlflow.log_metrics(metrics)
            mlflow.log_params({
                "label": label,
                "model_type": model_name,
                "max_depth": depth,
                "positive_customers": int(y.sum()),
                "cv_folds": folds.get_n_splits(),
                "cutoff_date": CUTOFF,
                "features_table": f"{DB}.gold_churn_training",
                "features_delta_version": features_version,
            })

            model.fit(X, y)
            # cloudpickle is set by hand because the newest MLflow default needs an extra package (skops)
            mlflow.sklearn.log_model(model, "model", input_example=X.head(5), serialization_format="cloudpickle")

            results.append({"run_id": run.info.run_id, "label": label, "model": model_name, **metrics})

2026/11/04 15:41:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c33d12a9-f75d-492b-81d6-b99d6d9fc688/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values `_ for more details.
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c33d12a9-f75d-492b-81d6-b99d6d9fc688/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inf

### 6.4 Compare the runs
The same comparison is in the MLflow UI: open Experiments, then `online_shop_churn`, tick the six runs from this execution and click Compare. This table shows the same numbers.

In [ ]:
comparison = pd.DataFrame(results).round(3)
print(comparison.drop(columns="run_id").to_string(index=False))

     label                model  accuracy  recall  precision    f1  roc_auc  customers_flagged
   churned  logistic_regression     0.960   0.000      0.000 0.000    0.224                 77
   churned random_forest_depth4     0.999   0.000      0.000 0.000    0.464                  0
   churned random_forest_depth8     0.999   0.000      0.000 0.000    0.471                  0
went_quiet  logistic_regression     0.662   0.819      0.093 0.167    0.804                729
went_quiet random_forest_depth4     0.707   0.711      0.095 0.168    0.791                621
went_quiet random_forest_depth8     0.836   0.434      0.114 0.180    0.774                317


### My choice: logistic regression on the `went_quiet` label

**On the brief's label, none of the three models is usable.** All three have a recall of 0, so they found neither of the 2 churned customers. The random forests show 99.9% accuracy only because they flag nobody at all.

**On `went_quiet` I pick logistic regression.** **Recall** matters most for a marketing team, because a customer we miss is a customer lost, while a wrong flag only costs one voucher or message. It has the best recall (0.819, so it finds 68 of the 83 customers) and the best ROC AUC (0.804).

**Its weakness is precision.** At 0.093 it flags 729 customers to find those 68, although that is still about twice as good as picking at random, since 4.2% of customers are in this group. Marketing should treat it as a cheap first filter and start with the highest scores. It mostly picks out customers who buy less often in general, because this data has no real pattern of people stopping.

In [ ]:
chosen = comparison[(comparison["label"] == "went_quiet") & (comparison["model"] == "logistic_regression")].iloc[0]
chosen_run_id = chosen["run_id"]
print("Chosen run ID:", chosen_run_id)
print(chosen.drop("run_id").to_string())

Chosen run ID: 4e63c9e7c82d4024ada9f25d1f1f158b
label                         went_quiet
model                logistic_regression
accuracy                           0.662
recall                             0.819
precision                          0.093
f1                                 0.167
roc_auc                            0.804
customers_flagged                    729


### 6.5 Reload the model by its run ID and score five customers
The model comes back from MLflow, not from the variable above. The columns must be in the same order as in training.

In [ ]:
loaded_model = mlflow.sklearn.load_model(f"runs:/{chosen_run_id}/model")

five = data.sample(5, random_state=7)
scored = five[["customer_id", "num_orders", "days_since_last_order", "orders_last_90_days"]].copy()
scored["risk_score"] = loaded_model.predict_proba(five[FEATURES])[:, 1].round(3)
scored["flag_for_marketing"] = scored["risk_score"] >= 0.5
scored["actual_orders_in_window"] = five["orders_in_window"]
print(scored.sort_values("risk_score", ascending=False).to_string(index=False))

 customer_id  num_orders  days_since_last_order  orders_last_90_days  risk_score  flag_for_marketing  actual_orders_in_window
         205          42                     27                    5       0.624                True                        5
        1759          40                      9                    5       0.531                True                        7
        1073          97                      5                   13       0.069               False                       11
        1049         188                      3                   28       0.000               False                       31
         695         244                      1                   55       0.000               False                       38
